# ARC_ATLAS_Test_v4

Evaluate trained checkpoints on local splits and downsampled variants.

- Loads training_v2.py, loads run config + best weights, builds inference model.
- Reads test cases from .../data/splits/80_20_random/test/{t1,masks}.
- Runs prediction per case, saves:
- *_pred_mask.nii.gz
- optional *_prob.nii.gz
- predictions.csv
- under RUN/test_predictions/<timestamp>/.

In [ ]:
from pathlib import Path
import csv
from dataclasses import fields
import importlib.util
import json
import time

import nibabel as nib
import numpy as np

PROJECT_ROOT = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
SRC = PROJECT_ROOT / 'src' / 'training_v2.py'
TEST_IMAGES_DIR = Path('/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/splits/80_20_random/test/t1')
TEST_MASKS_DIR = Path('/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/splits/80_20_random/test/masks')
TEST_ROOT = TEST_IMAGES_DIR.parent
if not TEST_IMAGES_DIR.exists():
    raise SystemExit(f'Test image dir not found: {TEST_IMAGES_DIR}')
if not TEST_MASKS_DIR.exists():
    print(f'Warning: mask dir not found: {TEST_MASKS_DIR} (will run image-only predictions)')
LIMIT_CASES = None
USE_TTA = True
SAVE_PROBS = True
THRESHOLD = None   # None -> per-case Otsu if enabled in config; else cfg.DECISION_THRESHOLD
MIN_COMPONENT_SIZE = 0
CLOSING_ITERS = 0

# Pick newest timestamped run that has config + best weights
def _resolve_latest_runnable_run() -> tuple[Path, Path]:
    run_dirs = sorted([p for p in (PROJECT_ROOT / 'runs').glob('20*') if p.is_dir()], reverse=True)
    for run in run_dirs:
        cfg = run / 'models' / 'config.json'
        w = run / 'callbacks' / 'best_model_dynamic.weights.h5'
        if cfg.exists() and w.exists():
            return run, w

    # Fallback path if only latest_best exists
    alt = PROJECT_ROOT / 'runs' / 'latest_best.weights.h5'
    latest_link = PROJECT_ROOT / 'runs' / 'latest'
    if alt.exists() and latest_link.exists() and (latest_link.resolve() / 'models' / 'config.json').exists():
        return latest_link.resolve(), alt

    raise SystemExit('No runnable run found under runs/ with models/config.json + best_model_dynamic.weights.h5')


RUN, WEIGHTS = _resolve_latest_runnable_run()
print('Using run:', RUN)
print('Weights:', WEIGHTS)

spec = importlib.util.spec_from_file_location('seg', SRC)
seg = importlib.util.module_from_spec(spec)
spec.loader.exec_module(seg)

cfg_json = json.load(open(RUN / 'models' / 'config.json'))
valid_cfg_keys = {f.name for f in fields(seg.DynamicTrainingConfig) if f.init}
cfg_json = {k: v for k, v in cfg_json.items() if k in valid_cfg_keys}
cfg_json['DATA_DIR'] = str(TEST_ROOT)
cfg_json['IMAGES_DIR'] = str(TEST_IMAGES_DIR)
cfg_json['MASKS_DIR'] = str(TEST_MASKS_DIR)
cfg = seg.DynamicTrainingConfig(**cfg_json)
cfg.MODEL_DIR = RUN / 'models'
model = seg.build_model_for_inference(cfg, weights_path=str(WEIGHTS))

print('Model input shape:', cfg.INPUT_SHAPE)
print('Patch size:', cfg.PATCH_SIZE)


def _resolve_path(raw_value: str, manifest_path: Path) -> Path | None:
    raw = (raw_value or '').strip()
    if not raw:
        return None
    p = Path(raw)
    if p.is_absolute():
        return p if p.exists() else None
    candidates = [p, TEST_ROOT / p, TEST_IMAGES_DIR / p]
    if TEST_MASKS_DIR.exists():
        candidates.append(TEST_MASKS_DIR / p)
    for base in manifest_path.parents:
        candidates.append(base / p)
    seen = set()
    for c in candidates:
        key = str(c)
        if key in seen:
            continue
        seen.add(key)
        if c.exists():
            return c
    return None


cases = []
manifest = TEST_ROOT / 'manifest.csv'
if manifest.exists():
    with open(manifest, newline='') as f:
        reader = csv.DictReader(f)
        for row in reader:
            t1 = _resolve_path(row.get('t1', ''), manifest)
            if t1 is None:
                continue
            msk = _resolve_path(row.get('mask', ''), manifest)
            key = row.get('key') or t1.stem
            cases.append({'key': key, 't1': t1, 'mask': msk if (msk and msk.exists()) else None})

if not cases:
    t1_dir = TEST_IMAGES_DIR
    msk_dir = TEST_MASKS_DIR
    for t1 in sorted(t1_dir.glob('*.nii.gz')):
        key = t1.name.replace('_T1w_MNI_norm.nii.gz', '').replace('.nii.gz', '')
        msk = None
        if msk_dir.exists():
            cand = msk_dir / t1.name.replace('_T1w_MNI_norm', '_lesion_mask_MNI_clean')
            if cand.exists():
                msk = cand
        cases.append({'key': key, 't1': t1, 'mask': msk})

if not cases:
    raise SystemExit(f'No test images found under {TEST_IMAGES_DIR}')

if LIMIT_CASES is not None:
    cases = cases[: int(LIMIT_CASES)]

pred_root = RUN / 'test_predictions'
pred_dir = pred_root / time.strftime('%Y%m%d_%H%M%S')
pred_dir.mkdir(parents=True, exist_ok=True)

rows = []
patch_size = tuple(cfg.PATCH_SIZE or cfg.INPUT_SHAPE[:-1])
for i, case in enumerate(cases, 1):
    img_obj = nib.as_closest_canonical(nib.load(str(case['t1'])))
    x = img_obj.get_fdata().astype(np.float32)

    probs = seg.gaussian_tta_predict(
        model,
        x,
        patch_size=patch_size,
        overlap=cfg.GAUSSIAN_TILE_OVERLAP,
        sigma=cfg.GAUSSIAN_TILE_SIGMA,
        tta=USE_TTA,
    )

    brain_mask = seg.compute_brain_mask(x)
    thr = THRESHOLD
    if thr is None and not bool(getattr(cfg, 'USE_PER_CASE_OTSU', True)):
        thr = float(getattr(cfg, 'DECISION_THRESHOLD', 0.1))

    pred = seg.apply_postprocessing(
        probs,
        threshold=thr,
        min_size=int(MIN_COMPONENT_SIZE),
        closing=int(CLOSING_ITERS),
        brain_mask=brain_mask,
        clamp=getattr(cfg, 'OTSU_CLAMP', (0.05, 0.25)),
        min_prob=float(getattr(cfg, 'OTSU_MIN_PROB', 0.01)),
    )
    pred_u8 = (pred > 0.5).astype(np.uint8)

    safe_key = str(case['key']).replace('/', '_').replace(' ', '_')
    pred_path = pred_dir / f"{safe_key}_pred_mask.nii.gz"
    nib.save(nib.Nifti1Image(pred_u8, img_obj.affine, img_obj.header), str(pred_path))

    prob_path = ''
    if SAVE_PROBS:
        prob_path = pred_dir / f"{safe_key}_prob.nii.gz"
        nib.save(nib.Nifti1Image(probs.astype(np.float32), img_obj.affine, img_obj.header), str(prob_path))

    row = {
        'key': case['key'],
        't1_path': str(case['t1']),
        'gt_mask_path': str(case['mask']) if case['mask'] else '',
        'pred_mask_path': str(pred_path),
        'prob_path': str(prob_path) if prob_path else '',
    }

    if case['mask'] and Path(case['mask']).exists():
        gt = nib.as_closest_canonical(nib.load(str(case['mask']))).get_fdata()
        y_true = (gt > 0.5).astype(np.float32)
        if y_true.shape != pred.shape:
            y_true = seg._center_crop_or_pad_volume(y_true, pred.shape)
        row['dice'] = float(seg.dice_soft_np(y_true, pred.astype(np.float32)))
    else:
        row['dice'] = ''

    rows.append(row)
    print(f"[{i}/{len(cases)}] {case['key']} -> {pred_path.name}" + (f" | dice={row['dice']:.4f}" if row['dice'] != '' else ''))

metrics_csv = pred_dir / 'predictions.csv'
with open(metrics_csv, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)

with_gt = [r for r in rows if r['dice'] != '']
if with_gt:
    mean_dice = float(np.mean([float(r['dice']) for r in with_gt]))
    print(f"Cases with GT masks: {len(with_gt)}/{len(rows)} | mean Dice: {mean_dice:.4f}")
else:
    print(f"No GT masks found. Generated predictions for {len(rows)} image(s).")
print('Prediction outputs:', pred_dir)
print('Summary CSV:', metrics_csv)



Using run: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/runs/20260225_145643
Weights: /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/runs/20260225_145643/callbacks/best_model_dynamic.weights.h5


2026-02-26 10:08:37.817879: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Visible GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
Mixed precision policy: <DTypePolicy "mixed_float16">
INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')


I0000 00:00:1772125719.702494  276970 gpu_process_state.cc:208] Using CUDA malloc Async allocator for GPU: 0
I0000 00:00:1772125719.703526  276970 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 17358 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:41:00.0, compute capability: 8.9
I0000 00:00:1772125719.703823  276970 gpu_process_state.cc:208] Using CUDA malloc Async allocator for GPU: 1
I0000 00:00:1772125719.704807  276970 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 17365 MB memory:  -> device: 1, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:61:00.0, compute capability: 8.9
2026-02-26 10:08:39,753 - SmartSOTA_Dynamic - INFO - ✅ All imports successful
2026-02-26 10:08:39,753 - SmartSOTA_Dynamic - INFO - TensorFlow eager execution: True
2026-02-26 10:08:39,754 - SmartSOTA_Dynamic - INFO - Environment verified:
- Python 3.10.18 (main, Jun  5 2025, 13:14:17) [GCC 11.2.0]
- TensorFlo

Strategy: MirroredStrategy
Model input shape: [112, 112, 112, 1]
Patch size: [112, 112, 112]


...

[4/104] sub-004 -> sub-004_pred_mask.nii.gz | dice=0.0795
[5/104] sub-005 -> sub-005_pred_mask.nii.gz | dice=0.0024
[6/104] sub-006 -> sub-006_pred_mask.nii.gz | dice=0.0723
[7/104] sub-007 -> sub-007_pred_mask.nii.gz | dice=0.0055


2026-02-26 10:16:11.010599: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node MultiDeviceIteratorGetNextFromShard}}]]
	 [[RemoteCall]]


2026-02-26 10:23:41.431122: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node MultiDeviceIteratorGetNextFromShard}}]]
	 [[RemoteCall]]


[15/104] sub-015 -> sub-015_pred_mask.nii.gz | dice=0.0167
[16/104] sub-016 -> sub-016_pred_mask.nii.gz | dice=0.0934
[17/104] sub-017 -> sub-017_pred_mask.nii.gz | dice=0.0096
[18/104] sub-018 -> sub-018_pred_mask.nii.gz | dice=0.0000
[19/104] sub-019 -> sub-019_pred_mask.nii.gz | dice=0.0061
[20/104] sub-020 -> sub-020_pred_mask.nii.gz | dice=0.0000
[21/104] sub-021 -> sub-021_pred_mask.nii.gz | dice=0.0646
[22/104] sub-022 -> sub-022_pred_mask.nii.gz | dice=0.0494
[23/104] sub-023 -> sub-023_pred_mask.nii.gz | dice=0.0029
[24/104] sub-024 -> sub-024_pred_mask.nii.gz | dice=0.0010


KeyboardInterrupt: 

In [ ]:
# --------- Optional fine-tune on Approx_Numeracy (domain adaptation) ---------
from pathlib import Path
from dataclasses import fields
import importlib.util
import json
import time
import traceback

PROJECT_ROOT = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
SRC = PROJECT_ROOT / 'src' / 'training_v2.py'

# Base-model selection for fine-tuning:
# 1) set BASE_RUN_OVERRIDE to pin a specific run,
# 2) or set USE_LATEST_SYMLINK=True to trust runs/latest,
# 3) otherwise use the newest timestamped run with config+weights.
BASE_RUN_OVERRIDE = None
USE_LATEST_SYMLINK = False


def _resolve_base_run() -> Path:
    if BASE_RUN_OVERRIDE:
        return Path(BASE_RUN_OVERRIDE).expanduser().resolve()

    latest_link = PROJECT_ROOT / 'runs' / 'latest'
    if USE_LATEST_SYMLINK and latest_link.exists():
        return latest_link.resolve()

    run_dirs = sorted([p for p in (PROJECT_ROOT / 'runs').glob('20*') if p.is_dir()], reverse=True)
    for run in run_dirs:
        if (run / 'models' / 'config.json').exists() and (run / 'callbacks' / 'best_model_dynamic.weights.h5').exists():
            return run

    raise SystemExit('No runnable run found under runs/ with models/config.json + callbacks/best_model_dynamic.weights.h5')


BASE_RUN = _resolve_base_run()
BASE_WEIGHTS = BASE_RUN / 'callbacks' / 'best_model_dynamic.weights.h5'
if not BASE_WEIGHTS.exists():
    raise SystemExit(f'Base weights not found: {BASE_WEIGHTS}')
if not SRC.exists():
    raise SystemExit(f'Training module not found: {SRC}')

spec = importlib.util.spec_from_file_location('seg', SRC)
if spec is None or spec.loader is None:
    raise SystemExit(f'Could not load module spec from {SRC}')
seg = importlib.util.module_from_spec(spec)
spec.loader.exec_module(seg)

FT_IMAGES_DIR = Path('/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/splits/80_20_random/test/t1')
FT_MASKS_DIR = Path('/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/splits/80_20_random/test/masks')
if not FT_IMAGES_DIR.exists() or not FT_MASKS_DIR.exists():
    raise SystemExit('Fine-tune folders missing. Check FT_IMAGES_DIR/FT_MASKS_DIR.')

# Conservative fine-tune schedule
FT_TOTAL_EPOCHS = 40
FT_EPOCH_STEPS = 300
FT_VAL_SPLIT = 0.20
FT_BATCH_SIZE = 2
FT_INITIAL_LR = 3e-6
FT_MIN_LR = 1e-7
FT_WARMUP_EPOCHS = 2
FT_COSINE_FIRST_CYCLE_EPOCHS = 12
FT_AUG_INTENSITY = 0.25
FT_ROTATION_RANGE = 10
FT_SYNTHETIC_LESION_PROB = 0.10

FT_RUN_ID = time.strftime('%Y%m%d_%H%M%S') + '_ft_approx'
FT_RUN_DIR = PROJECT_ROOT / 'runs' / FT_RUN_ID
FT_MODEL_DIR = FT_RUN_DIR / 'models'
FT_CALLBACKS_DIR = FT_RUN_DIR / 'callbacks'
FT_MODEL_DIR.mkdir(parents=True, exist_ok=True)
FT_CALLBACKS_DIR.mkdir(parents=True, exist_ok=True)

cfg_json = json.load(open(BASE_RUN / 'models' / 'config.json'))
valid_cfg_keys = {f.name for f in fields(seg.DynamicTrainingConfig) if f.init}
cfg_json = {k: v for k, v in cfg_json.items() if k in valid_cfg_keys}
cfg_json.update({
    'DATA_DIR': str(FT_IMAGES_DIR.parent),
    'IMAGES_DIR': str(FT_IMAGES_DIR),
    'MASKS_DIR': str(FT_MASKS_DIR),
    'MODEL_DIR': str(FT_MODEL_DIR),
    'CALLBACKS_DIR': str(FT_CALLBACKS_DIR),
    'TOTAL_EPOCHS': FT_TOTAL_EPOCHS,
    'INITIAL_EPOCH': 0,
    'EPOCH_STEPS': FT_EPOCH_STEPS,
    'VALIDATION_SPLIT': FT_VAL_SPLIT,
    'BATCH_SIZE': FT_BATCH_SIZE,
    'INITIAL_LR': FT_INITIAL_LR,
    'MIN_LR': FT_MIN_LR,
    'WARMUP_EPOCHS': FT_WARMUP_EPOCHS,
    'COSINE_FIRST_CYCLE_EPOCHS': FT_COSINE_FIRST_CYCLE_EPOCHS,
    'AUGMENTATION_INTENSITY': FT_AUG_INTENSITY,
    'ROTATION_RANGE': FT_ROTATION_RANGE,
    'SYNTHETIC_LESION_PROB': FT_SYNTHETIC_LESION_PROB,
    'DIFF_AWARE_ENABLED': False,
})

print('Base run:', BASE_RUN)
print('Base weights:', BASE_WEIGHTS)
print('Fine-tune run dir:', FT_RUN_DIR)
print('Expected output config:', FT_MODEL_DIR / 'config.json')
print('Expected output best weights:', FT_CALLBACKS_DIR / 'best_model_dynamic.weights.h5')
print('Expected output latest weights:', FT_CALLBACKS_DIR / 'latest.weights.h5')

try:
    ft_history = seg.train_dynamic_model(
        LOAD_WEIGHTS_FROM=str(BASE_WEIGHTS),
        RESUME_FROM_LATEST=False,
        **cfg_json,
    )
    print('Fine-tune complete. Keys:', list(getattr(ft_history, 'history', {}).keys()))
    print('Artifacts saved to', FT_RUN_DIR)
    print('Saved config:', FT_MODEL_DIR / 'config.json')
    print('Saved best weights:', FT_CALLBACKS_DIR / 'best_model_dynamic.weights.h5')
    print('Saved latest weights:', FT_CALLBACKS_DIR / 'latest.weights.h5')
except Exception:
    traceback.print_exc()
    raise




## (Optional) Evaluate on downsampled sets
Use helper functions from `src/downsampling/*.py` to generate degraded test sets in `data/downsampled/`, then load them with the training loader for metrics.